In [565]:
import pandas as pd
import jieba
import re
import wordcloud
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer  
import numpy as np
from sklearn.feature_extraction.text import TfidfTransformer  
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings("ignore")
from sklearn.cluster import KMeans
from sklearn.decomposition import LatentDirichletAllocation
import mglearn

In [566]:
data_2023=pd.read_excel("./low_2023.xlsx")

In [567]:
data_2023.head()

,mid,日期,内容,转发数,评论数,点赞数,博主,来自,uid,是否已认证,认证内容
0,4985123764965953,2023-12-31 21:33,湛女士一直都很希望能够拥有自己的孩子，但是却发现自己并没有怀运的能力。因此，她决定做三代试管...,NaN,NaN,NaN,小怀妈妈送好孕,微博网页版,2199823755,False,NaN
1,4985095582126892,2023-12-31 19:41,苗女士在二十六岁的时候就开始出现一些怪异的身体现象。医生告诉她这是因为她的卵巢衰退了。这意味...,NaN,NaN,NaN,小怀妈妈送好孕,微博网页版,2199823755,False,NaN
2,4985054528799643,2023-12-31 16:58,如果是不孕不育的问题不太严重，可以先尝试在国内做第一代、第二代试管，如果失败了再考虑去俄罗斯...,NaN,NaN,NaN,胖爸爸美国试管婴儿上海,微博网页版,7677311178,True,海外团队，跨境试管生殖机构咨询顾问
3,4985052263875826,2023-12-31 16:49,南京女客人和先生毕业就结婚了，因为年轻没想着要孩子，结婚3年后，想要孩子结果备孕了两年一直没...,NaN,NaN,NaN,胖爸爸美国俄罗斯泰国试管婴儿,微博网页版,3340004800,True,海外专家团队 专注试管生殖领域 迎接新生传递爱
4,4985044102285640,2023-12-31 16:16,绝育复通术后，促排三个月，成功怀孕，每天都有好消息，快来点赞接好孕呀！#不孕不育##试管婴儿...,NaN,NaN,NaN,宝宝梦工场周红医生,微博视频号,1338696775,True,北京五洲妇儿医院生殖科主任，今日排行榜签约自媒体人 母婴育儿博主


In [568]:
data_2023.shape

(5842, 11)

In [569]:
data_2023=data_2023[["内容"]]

In [570]:
data_2023.head()

,内容
0,湛女士一直都很希望能够拥有自己的孩子，但是却发现自己并没有怀运的能力。因此，她决定做三代试管...
1,苗女士在二十六岁的时候就开始出现一些怪异的身体现象。医生告诉她这是因为她的卵巢衰退了。这意味...
2,如果是不孕不育的问题不太严重，可以先尝试在国内做第一代、第二代试管，如果失败了再考虑去俄罗斯...
3,南京女客人和先生毕业就结婚了，因为年轻没想着要孩子，结婚3年后，想要孩子结果备孕了两年一直没...
4,绝育复通术后，促排三个月，成功怀孕，每天都有好消息，快来点赞接好孕呀！#不孕不育##试管婴儿...


In [571]:
data_2023=data_2023.drop_duplicates()

In [572]:
data_2023.head()

,内容
0,湛女士一直都很希望能够拥有自己的孩子，但是却发现自己并没有怀运的能力。因此，她决定做三代试管...
1,苗女士在二十六岁的时候就开始出现一些怪异的身体现象。医生告诉她这是因为她的卵巢衰退了。这意味...
2,如果是不孕不育的问题不太严重，可以先尝试在国内做第一代、第二代试管，如果失败了再考虑去俄罗斯...
3,南京女客人和先生毕业就结婚了，因为年轻没想着要孩子，结婚3年后，想要孩子结果备孕了两年一直没...
4,绝育复通术后，促排三个月，成功怀孕，每天都有好消息，快来点赞接好孕呀！#不孕不育##试管婴儿...


In [573]:
def remove_hashtags(text):
    # 使用正则表达式匹配两个#之间的内容，并替换为空字符串
    cleaned_text = re.sub(r'#(.*?)#', '', text)
    return cleaned_text

In [574]:
#删除话题标志
data_2023["内容"]=data_2023["内容"].apply(lambda x:remove_hashtags(x))

In [575]:
#用大模型判断是否是不孕不育话题
import requests
def send_post(url,data):
    try:
        response = requests.post(url, json=data)
        if response.status_code == 200:
            #             print("POST request was successful!")
            #             print("Response data:")
            return response.json()
        else:
            print(f"POST request failed with status code: {response.status_code}")
    except requests.exceptions.RequestException as e:
        print(f"Error occurred: {e}")
url="http://180.76.49.225:7861/chat"

In [576]:
# realt_flag=[]

In [577]:
for cont in data_2023["内容"].values:
    data={"question":"请帮我判断以下文本是否关于不孕不育或者生育的话题，如果是，请直接输出一个字是，否则直接输出不是，务必不要返回其他内容。文本是："+cont}
    realt_flag.append(send_post(url,data)["answer"])

In [578]:
data_2023["relat"]=realt_flag

In [579]:
data_2023=data_2023[data_2023["relat"].apply(lambda x:x=="是")]

In [580]:
data_2023.to_excel("./2023_rel.xlsx")

In [581]:
data_2023.shape

(4708, 2)

In [582]:
#替换名称
cn_city=["四川省","四川","南岳","上海","广州","深圳","东莞","重庆","北京","合肥","郑州","天津","无锡","南宁","忻州","长沙","南宁",
         "南王","漕河镇","黄冈","湖南","十安","湖南","十安","南京","信阳","沈阳","台湾","汕头","广东","河北","郴州","邯郸","海口",
        "河南省","兰州","杭州","济南","冀南","京沪","抚州","石家庄","洛阳市","洛阳","长春"]
out_city=["法国","泰国","澳洲","乌克兰","曼谷","马来西亚","英国政府","美国","欧洲","阿德莱德","俄罗斯"]
person=["顾岩","田纪香","王丽敏","王铁梅","铁梅","庹有烈","匡淑杰","田平君","田平","医羽","爱问","张潇潇","李媛","崔娟","郑芬芬",
       "李颖","周薇","董曦","程玲","白舰波","斯托克"]
hospi=["怡康","九院","百佳","宝来佳运","恒健","济爱","宜正堂","卫人","仁和","北辰","朝阳医院","衍宗堂","和睦家","仁爱医院","华博",
      "红房子","天伦","谷方益","同济","望京","京科","桃清堂","谷医堂"]

In [583]:
def replace(ss):
    for c in cn_city:
        ss=ss.replace(c,"国内")
    for c in out_city:
        ss=ss.replace(c,"国外")
    for c in person:
        ss=ss.replace(c,"医生")
    for c in hospi:
        ss=ss.replace(c,"医院")
    return ss

In [584]:
data_2023["内容"]=data_2023["内容"].apply(lambda x:replace(x))

In [585]:
#只保留中文
data_2023["content2"]=data_2023["内容"].apply(lambda x:re.sub(r'[^\u4e00-\u9fa5]','',str(x)))

In [586]:
#结巴分词
data_2023["words"]=data_2023["content2"].apply(lambda x:jieba.lcut(x))

In [587]:
data_2023.head()

,内容,relat,content2,words
0,湛女士一直都很希望能够拥有自己的孩子，但是却发现自己并没有怀运的能力。因此，她决定做三代试管...,是,湛女士一直都很希望能够拥有自己的孩子但是却发现自己并没有怀运的能力因此她决定做三代试管试试不...,"[湛, 女士, 一直, 都, 很, 希望, 能够, 拥有, 自己, 的, 孩子, 但是, 却..."
1,苗女士在二十六岁的时候就开始出现一些怪异的身体现象。医生告诉她这是因为她的卵巢衰退了。这意味...,是,苗女士在二十六岁的时候就开始出现一些怪异的身体现象医生告诉她这是因为她的卵巢衰退了这意味着她...,"[苗, 女士, 在, 二十六岁, 的, 时候, 就, 开始, 出现, 一些, 怪异, 的, ..."
2,如果是不孕不育的问题不太严重，可以先尝试在国内做第一代、第二代试管，如果失败了再考虑去国外或...,是,如果是不孕不育的问题不太严重可以先尝试在国内做第一代第二代试管如果失败了再考虑去国外或者国外...,"[如果, 是, 不孕, 不育, 的, 问题, 不太, 严重, 可以, 先, 尝试, 在, 国..."
3,国内女客人和先生毕业就结婚了，因为年轻没想着要孩子，结婚3年后，想要孩子结果备孕了两年一直没...,是,国内女客人和先生毕业就结婚了因为年轻没想着要孩子结婚年后想要孩子结果备孕了两年一直没有成功喝...,"[国内, 女, 客人, 和, 先生, 毕业, 就, 结婚, 了, 因为, 年轻, 没, 想着..."
4,绝育复通术后，促排三个月，成功怀孕，每天都有好消息，快来点赞接好孕呀！ 2国内·国内五洲妇儿...,是,绝育复通术后促排三个月成功怀孕每天都有好消息快来点赞接好孕呀国内国内五洲妇儿医院宝宝梦工场周...,"[绝育, 复通, 术后, 促排, 三个, 月, 成功, 怀孕, 每天, 都, 有, 好消息,..."


In [620]:
#读停用词
StopDictionary_path = './stop_words.txt'
stop_words = []

In [621]:
# 加载停用词
def get_stopwords(filePath):
    global stop_words
    f = open(filePath, 'r', encoding='utf8', errors='ignore')
    line = f.readline()
    while line:
        stop_words.append(line.strip())
        line = f.readline()
    f.close()

In [622]:
get_stopwords(StopDictionary_path)

In [623]:
#去掉数据中的停用词
def drop_stop_words(arr):
    word_clean=[]
    for w in arr:
        if len(w)>=2 and w not in stop_words:
            word_clean.append(w)
    return word_clean

In [624]:
data_2023["words_clean"]=data_2023["words"].apply(lambda x:drop_stop_words(x))

In [625]:
data_2023.head()

,内容,relat,content2,words,words_clean,words_str
0,湛女士一直都很希望能够拥有自己的孩子，但是却发现自己并没有怀运的能力。因此，她决定做三代试管...,是,湛女士一直都很希望能够拥有自己的孩子但是却发现自己并没有怀运的能力因此她决定做三代试管试试不...,"[湛, 女士, 一直, 都, 很, 希望, 能够, 拥有, 自己, 的, 孩子, 但是, 却...","[希望, 拥有, 孩子, 发现自己, 能力, 三代, 试管, 试试, 不幸, 子宫, 发育不...",希望 拥有 孩子 发现自己 能力 三代 试管 试试 不幸 子宫 发育不全 第一次 尝试 成功...
1,苗女士在二十六岁的时候就开始出现一些怪异的身体现象。医生告诉她这是因为她的卵巢衰退了。这意味...,是,苗女士在二十六岁的时候就开始出现一些怪异的身体现象医生告诉她这是因为她的卵巢衰退了这意味着她...,"[苗, 女士, 在, 二十六岁, 的, 时候, 就, 开始, 出现, 一些, 怪异, 的, ...","[二十六岁, 怪异, 身体, 现象, 医生, 告诉, 卵巢, 衰退, 意味着, 永远, 失望...",二十六岁 怪异 身体 现象 医生 告诉 卵巢 衰退 意味着 永远 失望 想要 孩子 听说 三...
2,如果是不孕不育的问题不太严重，可以先尝试在国内做第一代、第二代试管，如果失败了再考虑去国外或...,是,如果是不孕不育的问题不太严重可以先尝试在国内做第一代第二代试管如果失败了再考虑去国外或者国外...,"[如果, 是, 不孕, 不育, 的, 问题, 不太, 严重, 可以, 先, 尝试, 在, 国...","[不孕, 不育, 不太, 尝试, 国内, 第一代, 第二代, 试管, 失败, 国外, 国外,...",不孕 不育 不太 尝试 国内 第一代 第二代 试管 失败 国外 国外 做代 一般来说 国外 ...
3,国内女客人和先生毕业就结婚了，因为年轻没想着要孩子，结婚3年后，想要孩子结果备孕了两年一直没...,是,国内女客人和先生毕业就结婚了因为年轻没想着要孩子结婚年后想要孩子结果备孕了两年一直没有成功喝...,"[国内, 女, 客人, 和, 先生, 毕业, 就, 结婚, 了, 因为, 年轻, 没, 想着...","[国内, 客人, 毕业, 结婚, 年轻, 想着, 孩子, 结婚, 想要, 孩子, 备孕, 两...",国内 客人 毕业 结婚 年轻 想着 孩子 结婚 想要 孩子 备孕 两年 成功 半年 中药 调...
4,绝育复通术后，促排三个月，成功怀孕，每天都有好消息，快来点赞接好孕呀！ 2国内·国内五洲妇儿...,是,绝育复通术后促排三个月成功怀孕每天都有好消息快来点赞接好孕呀国内国内五洲妇儿医院宝宝梦工场周...,"[绝育, 复通, 术后, 促排, 三个, 月, 成功, 怀孕, 每天, 都, 有, 好消息,...","[绝育, 复通, 术后, 促排, 三个, 成功, 怀孕, 好消息, 快来点, 赞接, 好孕,...",绝育 复通 术后 促排 三个 成功 怀孕 好消息 快来点 赞接 好孕 国内 国内 妇儿 医院...


In [626]:
data_2023["words_str"]=data_2023["words_clean"].apply(lambda x:" ".join(x))

In [627]:
#将文本中的词语转换为词频矩阵  
vectorizer = CountVectorizer()  
#计算个词语出现的次数  
X = vectorizer.fit_transform(data_2023["words_str"])  
#获取词袋中所有文本关键词  
word = vectorizer.get_feature_names()  
print (word)
#查看词频结果  
print(X.toarray())

['一万', '一上午', '一下子', '一不爱', '一丝', '一两个', '一两周', '一两天', '一两年', '一个个', '一个劲', '一个半月', '一个多月', '一个巴掌拍不响', '一个月', '一个点', '一个男孩', '一个遍', '一为', '一举', '一二', '一产个', '一人', '一代', '一件', '一份', '一会', '一会儿', '一位', '一住', '一体', '一例', '一侧', '一促', '一倍', '一做', '一做式', '一儿', '一共', '一关', '一具', '一册', '一再', '一击', '一刀', '一分', '一分收获', '一分耕耘', '一分钱', '一切正常', '一切都在', '一切都是', '一切顺利', '一利有', '一到', '一刹那', '一刻', '一刻起', '一剂', '一副', '一功', '一动不动', '一劳永逸', '一医', '一千', '一千多万', '一千多公里', '一半', '一半左右', '一去不复返了', '一取', '一口', '一口气', '一句', '一句句', '一只', '一台', '一号', '一同', '一名', '一向', '一听', '一员', '一周', '一味', '一哄而上', '一嘴', '一回', '一团', '一圈', '一场', '一坐', '一块', '一堆', '一声', '一备', '一大', '一大半', '一大堆', '一大早', '一大步', '一大笔', '一大部分', '一天到晚', '一天天', '一头雾水', '一套', '一女', '一好', '一孩', '一孩二孩三孩', '一定量', '一家', '一家人', '一家子', '一家老小', '一对', '一对一', '一封', '一封信', '一小', '一小部分', '一少', '一层', '一屋', '一岁', '一己', '一帆风顺', '一年', '一年之计在于春', '一年四季', '一并', '一度', '一座', '一张', '一心', '一忍', '一念之差', '一性', '一患', '一成', '一成不变', '一手', '一批', '一把', '一抱', '一拖再拖', '一招', '一排', '一搏'

In [628]:
#词的维度
print(len(word))

21198


In [629]:
#文本表述维度
X.toarray().shape

(4708, 21198)

In [630]:
#类调用  
transformer = TfidfTransformer()  
#将词频矩阵X统计成TF-IDF值  
tfidf = transformer.fit_transform(X)  
#查看数据结构 tfidf[i][j]表示i类文本中的tf-idf权重  
print(tfidf.toarray()  )

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [631]:
#查看第一个文本的词的tfidf值
for i in tfidf.toarray()[0]:
    if i!=0:
        print(i)

0.13196595832138708
0.18033737428775068
0.18688291303535123
0.13519560136531228
0.13072405876468576
0.21543382972256797
0.17165228474257088
0.17756609148260583
0.2670674391286966
0.11998694032367366
0.19547101572616693
0.1750471085424685
0.07870817052634371
0.0952823622138798
0.18688291303535123
0.13196595832138708
0.10875972557323019
0.18183448043084993
0.15467559440867196
0.19306169518129337
0.16784080808256677
0.12543379301940358
0.14231803011678762
0.16768986264904048
0.20102753081716063
0.21225474556760413
0.16114432390143996
0.1514142152940957
0.19306169518129337
0.16862858707544565
0.11901527155238072
0.12953547635506946
0.1405309766268625
0.0757536747652355
0.1908453351288816
0.24267501070435832


In [632]:
k=3

In [633]:
#LDA主题建模
#设置主题数量，学习方式，超参数α，β取默认值。
lda = LatentDirichletAllocation(n_components=k, learning_method='batch', max_iter=50, random_state=0)
lda_topics = lda.fit_transform(tfidf.toarray())

In [634]:
#对主题词做降序排列
sorting = np.argsort(lda.components_, axis=1)[:, ::-1]
feature_names = np.array(vectorizer.get_feature_names())

mglearn.tools.print_topics(topics=range(k), feature_names=feature_names,
                           sorting=sorting, topics_per_chunk=5, n_words=20)

topic 0       topic 1       topic 2       
--------      --------      --------      
国内            国内            客人            
烧香            医院            国外            
阿姨            不孕            宝宝            
日记            阳光            试管            
前列腺炎          怀孕            赴美            
早泄            输卵管           移植            
阳痿            不育            生命            
祈福            子宫            期待            
烧香拜佛          医生            可爱            
大庙            治疗            幸福            
肾虚            女性            宝妈            
失眠            检查            爱心            
攻略            卵巢            试管婴儿          
注意事项          试管            咨询中心          
菩萨            备孕            单身            
求子            试管婴儿          妈妈            
朋友            时间            顺利            
调理            内膜            出生            
心愿            妇科            家庭            
加持            导致            生子            




In [564]:
for k in range(3,7):
    print("============"+str(k)+"============")
    #LDA主题建模
    #设置主题数量，学习方式，超参数α，β取默认值。
    lda = LatentDirichletAllocation(n_components=k, learning_method='batch', max_iter=50, random_state=0)
    lda_topics = lda.fit_transform(tfidf.toarray())
    #对主题词做降序排列
    sorting = np.argsort(lda.components_, axis=1)[:, ::-1]
    feature_names = np.array(vectorizer.get_feature_names())

    mglearn.tools.print_topics(topics=range(k), feature_names=feature_names,
                               sorting=sorting, topics_per_chunk=5, n_words=20)

============3============
topic 0       topic 1       topic 2       
--------      --------      --------      
国内            试管            不孕            
阳光            客人            子宫            
医院            国内            女性            
输卵管           宝宝            不育            
不孕            国外            治疗            
妈妈            女士            检查            
主任            成功            怀孕            
医生            赴美            卵巢            
时间            孩子            导致            
怀孕            移植            内膜            
好孕            试管婴儿          输卵管           
就诊            顺利            影响            
复通            生命            试管婴儿          
粘连            烧香            医院            
未孕            幸福            卵泡            
宝宝            出生            精子            
怀上            可爱            月经            
主诊            三代            国内            
手术            妈妈            原因            
收起            期待            情况            


============4============
